In [1]:
import os
os.chdir(r"C:\vscode\graph-rag\Source")

In [2]:
%pwd

'C:\\vscode\\graph-rag\\Source'

In [3]:
from config.settings_loader import load_config

config = load_config("config/config.yaml")

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import json

chunks = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config["chunking"]["chunk_size"],
    chunk_overlap=config["chunking"]["chunk_overlap"],
    length_function=len,
    separators=[". ", "."]
)

# Process both volumes
volumes = [
    ("volume_1", config["data_source"]["jsonified_data"]["volume_1"]),
    ("volume_2", config["data_source"]["jsonified_data"]["volume_2"])
]

for volume_name, file_path in volumes:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Convert to Documents and split
    documents = [
        Document(page_content=entry['text'], metadata=entry['metadata'])
        for entry in data
    ]
    
    volume_chunks = text_splitter.split_documents(documents)
    chunks.extend(volume_chunks)
    
    print(f"Original entries for {volume_name}: {len(data)}")
    print(f"New chunks for {volume_name}: {len(volume_chunks)}")

print(f"\nTotal chunks: {len(chunks)}")

Original entries for volume_1: 132
New chunks for volume_1: 1096
Original entries for volume_2: 150
New chunks for volume_2: 1374

Total chunks: 2470


In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os

load_dotenv()

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    model=config["embedding"]["embedding_model"],
    api_key=os.getenv("OPENAI_API_KEY")
)

# Create FAISS vector store from chunks
vector_store = FAISS.from_documents(chunks, embeddings)

# Save to disk
vector_store_path = config["embedding"]["vector_store_path"]
vector_store.save_local("C:/vscode/graph-rag/Source/data/vector_store")

print(f"Vector store created and saved to {vector_store_path}")
print(f"Total vectors stored: {len(chunks)}")

Vector store created and saved to dataector_store
Total vectors stored: 2470


In [17]:
import os

# Ensure correct path format
vector_store_path = os.path.join(os.getcwd(), config["embedding"]["vector_store_path"])
print(f"Loading from: {vector_store_path}")  # Debug: check the actual path

loaded_vector_store = FAISS.load_local(
    vector_store_path,
    embeddings,
    allow_dangerous_deserialization=True
)

Loading from: C:\vscode\graph-rag\Source\dataector_store


RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\third-party\faiss\faiss\impl\io.cpp:70: Error: 'f' failed: could not open C:\vscode\graph-rag\Source\dataector_store\index.faiss for reading: Invalid argument

In [ ]:
# Alternative: Direct similarity search with scores
query = "What is the main topic discussed?"
docs_with_scores = loaded_vector_store.similarity_search_with_score(query, k=5)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(docs_with_scores, 1):
    print(f"--- Document {i} (Score: {score:.4f}) ---")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}\n")